In [1]:

from __future__ import division         # for Python 2 compatibility
from __future__ import print_function   # for Python 2 compatibility
from pathlib import Path                # treat paths as objects with methods instead of strings

import matplotlib.pyplot as plt         # import of matplotlib
import numpy as np                      # import of numpy
import pandas as pd                     # data manipulation library
import glob                             # search folders and files on the computer
import os                               # built-in library for system
import openpyxl                         # read excel data
from openpyxl import load_workbook
%matplotlib inline
# renders images as PNG inside VSC

# !
# ?
# *
# TODO
# // 

last update: 24/11/26 <br>
Goal: update previously generated result file to overview file
- Remove all sheets except the overview 
- Add separate sheets for SB and each sample type 
- Calculate average and LOD for SB sheet and add values to all sample sheets
- Calculate corrected ppb values for all sample sheets
- Create new sheet 'ppb' with an overview of all corrected ppb values (sorted), move this sheet to second place and remove columns B-C
- Calculate average and SDev of corrected ppb values for all sample sheets

In [ ]:
folder = r"C:\yourfolderhere"              #* Define the folder containing the Excel file - for all following activities
result_path = os.path.join(folder, 'Results.xlsx')               #* Define the path for the new Excel overview file - for all following activities

overview_file = 'Overview.xlsx'
overview_path = os.path.join(folder, overview_file)

# Delete all sheets except the first one
wb = load_workbook(result_path) 
sheet_names = wb.sheetnames
for sheet in sheet_names[1:]:           #* Skip the first sheet
    wb.remove(wb[sheet])
wb.save(overview_path)

# Add SB rows to a new sheet 'SB', don't move 'CB'
result_sheets = pd.read_excel(result_path, sheet_name=None, engine='openpyxl')    #* Reload all updated sheets with avg values into a dictionary of DataFrames
first_sheet = list(result_sheets.keys())[0]                                       #* Access the first sheet
data = result_sheets[first_sheet]
first_column = data.iloc[:, 0]                                                    #* Select the first column

# Create new individual sheets for all sample types
sample_type = first_column.unique()                                         #* Retrieve all unique values from first column (sample types)

with pd.ExcelWriter(overview_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:  #* Open file in append mode to add new sheets
    for value in sample_type:
        if value == 'CB':                                                   #* Skip CB rows and dont create new sheet
            continue
        filtered_data = data[first_column == value ]                        #* Filter for rows with the same unique value in the first column
        sheet_name = str(value)                                             #* Convert value to string and use it as sheet name
        filtered_data.to_excel(writer, sheet_name=sheet_name, index=False)  #* Add filtered data to a new sheet

print("Updated overview file with individual sheets for SB and all sample types.")

Updated overview file with individual sheets for SB and all sample types.


In [ ]:
# Reorder sheets so SB is the second sheet 
wb = load_workbook(overview_path)

if 'Overview' and 'SB' in wb.sheetnames:
    sheet_names = wb.sheetnames
    sheet_names.remove('Overview')
    sheet_names.remove('SB')
    wb._sheets = [wb['Overview']] + [wb['SB']] + [wb[sheet] for sheet in sheet_names]

wb.save(overview_path)
print("SB sheet moved to the second position.")

In [3]:
#Calculate LOD and Average for SB sheet
#AVG =IF(AVERAGE(C2:C8)>0,AVERAGE(C2:C8),0)
#LOD =3*STDEV.S(C2:C8)

SB_sheet = pd.read_excel(overview_path, sheet_name='SB')              #* Loads SB sheet into a new data frame

#Update sheets to overview file
with pd.ExcelWriter(overview_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:    
    if len(SB_sheet) >= 3:                                                                  #! Adjust this if you have less than 3 SB
        SB_average = SB_sheet.mean(axis=0, numeric_only=True)                               #* Calculate the mean of each column (axis=0), header is ignored
        SB_average = pd.DataFrame(SB_average).T                                             #* Transpose mean values into data frame
        
        SB_LOD = 3*(SB_sheet.std(axis=0, ddof=1, numeric_only=True))                        #* Calculate the LOD of each column (ddof=1 specifies division by n-1 in stddev formula)
        SB_LOD = pd.DataFrame(SB_LOD).T                                                     #* Transpose LOD values into data frame
        
        empty_row = pd.DataFrame([['']*len(SB_sheet.columns)], columns=SB_sheet.columns)    #* Add an empty row

        # Create new data frame where the average and LOD values are added as a new row, with one blank row in between (concat = combine)
        SB_calc = pd.concat([SB_sheet, empty_row, SB_average, SB_LOD], ignore_index=True)
        SB_values = pd.concat([empty_row, SB_average, SB_LOD], ignore_index=True)           #* Needed for following analysis

        # Add respective labels directly to column A
        SB_calc.at[len(SB_sheet)+1, SB_calc.columns[0]] = 'Average SB'                      #* Label for the mean row
        SB_calc.at[len(SB_sheet)+2, SB_calc.columns[0]] = 'LOD SB'                          #* Label for the LOD row
        
        SB_calc.to_excel(writer, sheet_name='SB', index=False)
        print(f"Averaged SB values added.")
    else:
        print(f"Less than 3 SB files detected. Skipping...")

print(f"Averaged values have been added to SB sheet in {overview_path}")

Averaged SB values added.
Averaged values have been added to SB sheet in C:\Users\install\Desktop\Ionicon\Viewer Projects\250730\Overview.xlsx


In [4]:
sheets = pd.read_excel(overview_path, sheet_name=None)                           #* Get all sheet names in the file
sample_sheets = { name:  sheets[name] for name in list(sheets.keys())[2:]}       #* Load all sample sheets into a new data frame (starting from third sheet)

#Add Average and LOD of SB to all sample sheets
with pd.ExcelWriter(overview_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    for sheet_name, df in sample_sheets.items():
        if len(df) >= 1:                                                        #* Check that at least one sample row is present
            # Create new data frame where the Average and LOD values from the SB are added as a new row, with one blank row in between (concat = combine)
            sample_calc = pd.concat([df, SB_values], ignore_index=True) 

            # Add respective labels directly to column A
            sample_calc.at[len(df)+1, sample_calc.columns[0]] = 'Average SB'    #* Label for the mean row
            sample_calc.at[len(df)+2, sample_calc.columns[0]] = 'LOD SB'        #* Label for the LOD row

            sample_calc.to_excel(writer, sheet_name=sheet_name, index=False)
            print(f"SB values added to {sheet_name}.")
        else:
            print(f"Less than 3 Sample files detected. Skipping...")

print(f"Average SB and LOD SB values have been added to all sample sheets.")

SB values added to one.
SB values added to FB.
SB values added to smone.
SB values added to twofive.
SB values added to SB.
Average SB and LOD SB values have been added to all sample sheets.


In [5]:
#'UPDATED' 24/11/26

sheets = pd.read_excel(overview_path, sheet_name=None)                          #* Get all names for the sheets in the file
sample_sheets = { name:  sheets[name] for name in list(sheets.keys())[2:]}      #* Loads sample sheets into a new data frame (starting from third sheet)

# Calculate corrected ppb values for further algorithm analysis 
with pd.ExcelWriter(overview_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    for sheet_name, df in sample_sheets.items():
        if len(df) >= 1:                                                        #* Check that at least one sample row is present

            # Find sample rows, Average SB row and LOD SB row
            first_empty_row = df[df.isnull().all(axis=1)].index.min()           #* Find the index of the first blank row (all columns are NaN or empty)
            print(f"Number of sample rows in {sheet_name}: {first_empty_row}")

            sample = df.iloc[:first_empty_row, 3:]                              #* Extract sample data (starting from fourth column)
            avg_SB = df.loc[df.iloc[:, 0] == 'Average SB', df.columns[3:]]        
            LOD_SB = df.loc[df.iloc[:, 0] == 'LOD SB', df.columns[3:]]                

            ppb_values = pd.DataFrame(index=sample.index, columns=sample.columns)       #* Create empty df for new ppb values (same index and columns)
            empty_rows = pd.DataFrame([['']*len(df.columns)]*2, columns=df.columns)     #* Create two empty rows

            # Iterate over each column to calculate ppb values
            for col in sample.columns:                                                  #* Iterate over each column in the sample data
                avg_SB_value = avg_SB.iloc[0][col]                                      #* Convert df to a series of values for further calculations
                LOD_SB_value = LOD_SB.iloc[0][col] 
                for row in sample.index:                                                #* Iterate over each row index in the sample data
                    ppb = sample.at[row,col] - avg_SB_value - LOD_SB_value              #* Calculate ppb for each column, using the average and LOD value from this column
                    if ppb >0:
                        ppb_values.at[row, col] = sample.at[row,col] - avg_SB_value
                    else :
                        ppb_values.at[row, col] = 0

            ppb_calc = pd.concat([df, empty_rows, ppb_values], ignore_index=False)      #* Create a new df by combining the current sheet, ppb_values, and empty rows

            # Add all sample names to the corrected ppb value rows, combining sample + No
            LOD_SB_index = df[df.iloc[:, 0] == 'LOD SB'].index[0]
            ppb_calc.iat[LOD_SB_index + 2, 0] = 'corrected ppb values'                  #* Label for the calculated values (position-based indexing)

            for i in range(len(sample)):                                                #* Iterate through all sample names
                row_index = LOD_SB_index + 3 + i                                        #* Starting row for the sample names

                if row_index < len(ppb_calc):                                           #* Ensure the row is within bounds of ppb_calc
                    value_col_1 = ppb_calc.iloc[i, 0]                                   #* Value from column 1 of sample (Sample name)
                    value_col_2 = ppb_calc.iloc[i, 1]                                   #* Value from column 2 of sample (Sample No.)
        
                    if pd.notna(value_col_2) and value_col_2 != '':                             #* If there's a value in column 2 (not NaN or empty),
                        ppb_calc.iat[row_index, 0] = f"{value_col_1}_{int(value_col_2)}"        #* concatenate values from column 1 and 2 with an underscore
                    else:
                        ppb_calc.iat[row_index, 0] = f"{value_col_1}"                           #* If there's no value in column 2, just append the value from column 1
                else:
                    print(f"Warning: row_index {row_index} is out of bounds for ppb_calc.")

            ppb_calc.to_excel(writer, sheet_name=sheet_name, index=False)
            print(f"Corrected ppb values added to {sheet_name}.")
        else:
            print(f"Some values missing in {sheet_name} . Skipping...")

print(f"Corrected ppb values have been added to all sample sheets.")

Number of sample rows in one: 3
Corrected ppb values added to one.
Number of sample rows in FB: 3
Corrected ppb values added to FB.
Number of sample rows in smone: 3
Corrected ppb values added to smone.
Number of sample rows in twofive: 3
Corrected ppb values added to twofive.
Number of sample rows in SB: 3
Corrected ppb values added to SB.
Corrected ppb values have been added to all sample sheets.


In [6]:
#'UPDATED 24/11/26'

# Create a new sheet 'ppb' with all the corrected ppb values
sheets = pd.read_excel(overview_path, sheet_name=None, engine='openpyxl')     #* Reload all updated sheets with avg values into a dictionary of DataFrames
sample_sheets = { name:  sheets[name] for name in list(sheets.keys())[2:]}    #* Load sample sheets into a new data frame (starting from third sheet)
ppb_values = []                                                               #* Create empty df for all ppb values

# Loop through the dictionary of sheets
for sheet_name, df in sample_sheets.items():
    if len(df) >= 1:                                                          
        header = df.columns.tolist()                                          #* Access the header

        ppb_index = df[df.iloc[:, 0] == 'corrected ppb values'].index[0]      #* Find the index of "corrected ppb values" row
        ppb_rows = df.iloc[ppb_index + 1:, :]                                 #* Extract all following rows

        ppb_values.append(ppb_rows)                                           #* Append ppb rows to df
        ppb_df = pd.concat(ppb_values, ignore_index=True)                     #* Create the combined ppb overview df
        print(f"Corrected ppb values from {sheet_name} added to new ppb sheet.")
    else:
        print(f"Sheet {sheet_name} does not have enough rows. Skipping...")

# Add the ppb sheet to the workbook
with pd.ExcelWriter(overview_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    ppb_df.to_excel(writer, sheet_name='ppb', index=False)

# Reorder rows based on sample type and No.
ppb_sheet = pd.read_excel(overview_path, sheet_name='ppb', engine='openpyxl')
ppb_sheet['sample_type_str'] = ppb_sheet.iloc[:, 0].str.extract(r'^([A-Za-z]+)')            #* Extract the text part before '_' (sample type)
ppb_sheet['sample_type_num'] = ppb_sheet.iloc[:, 0].str.extract(r'(\d+)').astype(float)     #* Extract the numeric part before '_' (sample type)
ppb_sheet['No'] = ppb_sheet.iloc[:, 0].str.extract(r'_(\d+)$').astype(float)                #* Extract the numeric part after '_' (sample No.)
ppb_sheet['sample_type_str'] = ppb_sheet['sample_type_str'].fillna('')                      #* Empty str for num prefix
ppb_sorted = ppb_sheet.sort_values(
    by=['sample_type_str', 'sample_type_num', 'No']
    ).reset_index(drop=True)                                                                #* Sort by 'sample type' (alphabetically) and 'No' (numerically)

with pd.ExcelWriter(overview_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    ppb_sorted.to_excel(writer, sheet_name='ppb', index=False)

print(f"New sheet 'ppb' with sorted ppb values added to {overview_path}.")

# Reorder sheets so ppb is the second sheet and remove columns B-K
wb = load_workbook(overview_path)

if 'Overview' and 'ppb' in wb.sheetnames:
    sheet_names = wb.sheetnames
    sheet_names.remove('Overview')
    sheet_names.remove('ppb')
    wb._sheets = [wb['Overview']] + [wb['ppb']] + [wb[sheet] for sheet in sheet_names]

ws = wb['ppb']
for col in range (11, 1, -1):        #* Note: columns are 1-indexed, so range is from 11 (start is inclusive) to 1 (stop is exclusive) (columns K-B)
    col_letter = chr(64 + col)       #* Convert column number to letter
    ws.delete_cols(col)

for col in range (659, 657, -1):     #* Note: columns are 1-indexed, so range is from 659 (start is inclusive) to 657 (stop is exclusive) (columns YH-YI)
    col_letter = chr(64 + col)       #* Convert column number to letter
    ws.delete_cols(col)

wb.save(overview_path)
print("ppb sheet moved to the second position and first ten and last two columns removed.")

Corrected ppb values from one added to new ppb sheet.
Corrected ppb values from FB added to new ppb sheet.
Corrected ppb values from smone added to new ppb sheet.
Corrected ppb values from twofive added to new ppb sheet.
Corrected ppb values from SB added to new ppb sheet.
New sheet 'ppb' with sorted ppb values added to C:\Users\install\Desktop\Ionicon\Viewer Projects\250730\Overview.xlsx.
ppb sheet moved to the second position and first ten and last two columns removed.


In [7]:
#'UPDATED' 24/11/26
sheets = pd.read_excel(overview_path, sheet_name=None)                           #* Get all names for the sheets in the file
sample_sheets = { name:  sheets[name] for name in list(sheets.keys())[3:]}       #* Load sample sheets into a new data frame (starting from fourth sheet)

#Generate Average and SDev for corrected ppb values
with pd.ExcelWriter(overview_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer: 
    for sheet_name, df in sample_sheets.items():
        if len(df) >= 1:
            # Select all rows with corrected ppb values
            ppb_rows = df.iloc[ppb_index + 1:, :]                                 

            # Calculate average and standard deviation
            avg_ppb = ppb_rows.mean(axis=0, numeric_only=True)                          #* Calculate the mean of each column (axis=0), header is ignored
            avg_ppb = pd.DataFrame(avg_ppb).T                                           #* Transpose mean values into data frame
            SDev_ppb = ppb_rows.std(axis=0, ddof=1, numeric_only=True)                  #* Calculate the SDev of each column (ddof=1 specifies division by n-1 in stddev formula)
            SDev_ppb = pd.DataFrame(SDev_ppb).T                                         #* Transpose SDev values into data frame

            empty_row = pd.DataFrame([['']*len(df.columns)], columns=df.columns)        #* Add an empty row

            # Create new data frame where the average and SDev values are added as a new row, with one blank row in between (concat = combine)
            ppb_calc = pd.concat([df, empty_row, avg_ppb, SDev_ppb], ignore_index=True)
            #sample_values = pd.concat([empty_row, average, SDev], ignore_index=True)

            # Add labels directly to column A
            ppb_calc.at[len(df)+1, ppb_calc.columns[0]] = 'Average Corr'       #* Label for the mean row
            ppb_calc.at[len(df)+2, ppb_calc.columns[0]] = 'SDev Corr'          #* Label for the SDev row

            ppb_calc.to_excel(writer, sheet_name=sheet_name, index=False)
            print(f"Averaged values and SDev added to {sheet_name}.")
        else:
            print(f"No rows detected in {sheet_name}. Skipping...")

    print(f"Average and SDev for corrected ppb values have been added to all sample sheets in {overview_path}.")

Averaged values and SDev added to one.
Averaged values and SDev added to FB.
Averaged values and SDev added to smone.
Averaged values and SDev added to twofive.
Averaged values and SDev added to SB.
Average and SDev for corrected ppb values have been added to all sample sheets in C:\Users\install\Desktop\Ionicon\Viewer Projects\250730\Overview.xlsx.


In [ ]:
# Generate a new info page
rows= [
    "E/N [Td]",
    "TD Flow (measured) [L/min]", 
    "PTR-MS Flow [L/min]",
    "Integration time [min]",
    "Sample Volume [mL]",
    ]
df = pd.DataFrame(rows, columns = ["Measurement information"])

with pd.ExcelWriter(overview_path, engine='openpyxl', mode='a', if_sheet_exists='new') as writer: 
    df.to_excel(writer, sheet_name='Info', index=False)